- Start with file: 'data_no_duplicates.parquet'

OBJECTIVE:
reduce dataset by:
- removing 'falling from vehicle' accidents
- removing accidents with 2 or more drivers
- removing accidents with no pedestrians involved
- removing accidents with no driver
- removing columns with a large amount of missing data (Airbag & Seatbelt/Helmet columns)

clean dataset by:
- removing progressivo number inconsistencies
- filling in / removing 'TipoPersona' values
- logging where parked vehicles are involved, removing 'In sosta' rows

- End with file: '004_data_only_pedestrians.parquet'

In [39]:
import pandas as pd
import numpy as np
import pyarrow

In [40]:
df = pd.read_parquet("data_no_duplicates.parquet").copy()
df.shape

(685877, 37)

The dataset currently has 37 columns and 685877 rows of data. 
Let's look at a couple of rows of data: 

In [41]:
df.head(2).T

,0,1
Protocollo,1931569,1931569
Gruppo,7,7
DataOraIncidente,01/01/2013 13:15,01/01/2013 13:15
Localizzazione1,Strada Urbana,Strada Urbana
STRADA1,VIA DEI CICLAMINI,VIA DEI CICLAMINI
Localizzazione2,all'intersezione con,all'intersezione con
STRADA2,VIA DELLE ROSE,VIA DELLE ROSE
Strada02,VIA DELLE ROSE,VIA DELLE ROSE
Chilometrica,,
DaSpecificare,,


In [42]:
# Examine the columns:
df.columns

Index(['Protocollo', 'Gruppo', 'DataOraIncidente', 'Localizzazione1',
       'STRADA1', 'Localizzazione2', 'STRADA2', 'Strada02', 'Chilometrica',
       'DaSpecificare', 'NaturaIncidente', 'particolaritastrade', 'TipoStrada',
       'FondoStradale', 'Pavimentazione', 'Segnaletica',
       'CondizioneAtmosferica', 'Traffico', 'Visibilita', 'Illuminazione',
       'NUM_FERITI', 'NUM_RISERVATA', 'NUM_MORTI', 'NUM_ILLESI', 'Longitude',
       'Latitude', 'Confermato', 'Progressivo', 'TipoVeicolo', 'StatoVeicolo',
       'TipoPersona', 'Sesso', 'Tipolesione', 'Deceduto', 'DecedutoDopo',
       'CinturaCascoUtilizzato', 'Airbag'],
      dtype='object')

It will be easier to understand the dynamics of each accident with the information in a different order. 
We can also drop the 'Confermato' column; this is a bureaucratic item and does not give us information about the accident. 

In [43]:
priority_columns = ['Protocollo',  'Progressivo', 'TipoPersona', 'TipoVeicolo', 'StatoVeicolo',
                    'NaturaIncidente',  'NUM_FERITI', 'NUM_RISERVATA', 'NUM_MORTI', 'NUM_ILLESI',
                    'Sesso']
time_columns = ['DataOraIncidente', 'CondizioneAtmosferica', 'Traffico']
position_columns = ['Gruppo', 'Localizzazione1', 'STRADA1', 'Localizzazione2',
                    'STRADA2', 'Strada02', 'Chilometrica', 'DaSpecificare', 'Latitude', 'Longitude']
injury_columns = ['CinturaCascoUtilizzato', 'Airbag',
                  'Tipolesione', 'Deceduto', 'DecedutoDopo']
road_type_columns = ['particolaritastrade', 'TipoStrada',
                     'FondoStradale', 'Pavimentazione', 'Segnaletica', 'Visibilita', 'Illuminazione']

df = df[priority_columns + time_columns +
        position_columns + road_type_columns + injury_columns]
df.shape

(685877, 36)

In [44]:
def analyse_column(column_name):
    """
    Analyzes value counts, percentages, and unique protocols for a column in the DataFrame.

    Args:
        column_name (str): The name of the column to analyze.

    Returns:
        pandas.DataFrame: A DataFrame with 'Count', '% per category', and 
                         'Unique Accidents' for each unique value.
    """

    # Validate input
    if column_name not in df.columns:
        raise ValueError(f"Column '{column_name}' not found in DataFrame")

    # Clean the column by replacing empty strings with NaN
    cleaned_column = df[column_name].replace('', pd.NA)

    # Calculate counts and percentages
    counts = cleaned_column.value_counts(dropna=False)
    percentages = (cleaned_column.value_counts(
        dropna=False, normalize=True) * 100).round(2)

    # Calculate unique protocols for each category
    unique_protocols = []
    for category in counts.index:
        if pd.isna(category):
            # Handle NaN values
            mask = cleaned_column.isna()
        else:
            # Handle non-NaN values
            mask = cleaned_column == category

        protocol_count = df.loc[mask, 'Protocollo'].nunique()
        unique_protocols.append(protocol_count)

    # Create result DataFrame
    result = pd.DataFrame({
        'Count': counts,
        '% per category': percentages,
        'Unique Accidents': unique_protocols
    }, index=counts.index)

    return result

In [45]:
analyse_column('NaturaIncidente')

,Count,% per category,Unique Accidents
NaturaIncidente,,,
Scontro laterale fra veicoli in marcia,169866,24.77,62593
Scontro frontale/laterale DX fra veicoli in marcia,91496,13.34,32683
Scontro frontale/laterale SX fra veicoli in marcia,88701,12.93,31996
Tamponamento,83167,12.13,30492
Tamponamento Multiplo,51942,7.57,11957
Investimento di pedone,41485,6.05,18204
Veicolo in marcia contro ostacolo accidentale,29808,4.35,21209
Veicolo in marcia contro veicolo in sosta,27251,3.97,15561
Veicolo in marcia contro ostacolo fisso,22031,3.21,15980


DATA CLEANING - DELETE AS MANY ACCIDENTS AS POSSIBLE - FALLING FROM VEHICLE

1) Delete accidents from category: Infortunio per caduta del veicolo (falling from vehicle)

In [46]:
print(f"Removing 'falling from a vehicle' accidents")
print(f"Shape before: {df.shape}")
df = df[df['NaturaIncidente'] !=
        'Infortunio per caduta del veicolo'].reset_index(drop=True)

print(f"Shape after: {df.shape}")

Removing 'falling from a vehicle' accidents
Shape before: (685877, 36)
Shape after: (682844, 36)


DATA CLEANING - DELETE AS MANY ACCIDENTS AS POSSIBLE - TWO OR MORE DRIVERS

2) Delete accidents with more than one driver

In [47]:
# Identify protocols with multiple drivers
multi_driver_protocols = (
    df.loc[df["TipoPersona"] == "Conducente", "Protocollo"]
      .value_counts()
      .loc[lambda x: x >= 2]
      .index
)

print(f"Removing {len(multi_driver_protocols)} multi-driver accidents")
print(f"Shape before: {df.shape}")

# Remove those protocols
df = df[~df["Protocollo"].isin(multi_driver_protocols)].reset_index(drop=True)

print(f"Shape after: {df.shape}")

Removing 183896 multi-driver accidents
Shape before: (682844, 36)
Shape after: (151400, 36)


DATA CLEANING - DELETE AS MANY ACCIDENTS AS POSSIBLE - NO PEDESTRIANS INVOLVED

3) Delete accidents with no pedestrians

In [48]:
# Find accidents involving at least one pedestrian
pedestrian_protocols = df.loc[df["TipoPersona"] == "Pedone", "Protocollo"]

print(f"Keeping {len(pedestrian_protocols)} pedestrian accidents")
print(f"Shape before: {df.shape}")

# Keep protocols involving one or more pedestrians
df = df[df["Protocollo"].isin(pedestrian_protocols)].reset_index(drop=True)

print(f"Shape after: {df.shape}")

Keeping 19730 pedestrian accidents
Shape before: (151400, 36)
Shape after: (41300, 36)


DATA CLEANING - DELETE AS MANY ACCIDENTS AS POSSIBLE - FIND 'PROGRESSIVO' INCONSISTENCIES

4) Identify accidents with inconsistent 'progressivo' numbers (each separate progressivo number denotes a separate vehicle) 

In [49]:
problem_protocols_progressivo = []

# Group by Protocollo
for protocollo, group in df.groupby("Protocollo"):

    progressivi = sorted(group["Progressivo"].dropna().astype(
        int).unique(), reverse=True)

    # Check if progressivi are sequential (highest to lowest)
    for i in range(len(progressivi) - 1):
        if progressivi[i] - progressivi[i + 1] != 1:
            problem_protocols_progressivo.append(protocollo)
            break  # No need to check further once a gap is found

df_problematic = df[df["Protocollo"].isin(problem_protocols_progressivo)]

In [50]:
df_problematic

,Protocollo,Progressivo,TipoPersona,TipoVeicolo,StatoVeicolo,NaturaIncidente,NUM_FERITI,NUM_RISERVATA,NUM_MORTI,NUM_ILLESI,...,FondoStradale,Pavimentazione,Segnaletica,Visibilita,Illuminazione,CinturaCascoUtilizzato,Airbag,Tipolesione,Deceduto,DecedutoDopo
23114,3763049,<NA>,Pedone,,,Veicolo in marcia contro veicoli in sosta,1,0,0,2,...,Asciutto,Asfaltata,Verticale,Buona,Ore Diurne,,,Illeso,0,
23115,3763049,1,Conducente,Autovettura privata,In marcia / fermata / arresto,Veicolo in marcia contro veicoli in sosta,1,0,0,2,...,Asciutto,Asfaltata,Verticale,Buona,Ore Diurne,,,Ricoverato,0,NON DECEDUTO
23116,3763049,3,Passeggero,Autovettura privata,Sosta,Veicolo in marcia contro veicoli in sosta,1,0,0,2,...,Asciutto,Asfaltata,Verticale,Buona,Ore Diurne,,,Illeso,0,


There is a single protocol affected. This protocol can be retained; a moving vehicle hit a parked vehicle and a pedestrian. Although there is a gap in the 'progressivo' number, we can simply change this progressivo 3 to 2, as there were clearly one moving vehicle and a stopped vehicle involved. 

In [51]:
df.loc[23116, 'Progressivo'] = 2
df.loc[23116, ['Protocollo', 'Progressivo']]

Protocollo     3763049
Progressivo          2
Name: 23116, dtype: object

DATA CLEANING - 'NATURAINCIDENTE' BLANK 

5) Delete / fix rows where the 'NaturaIncidente' (accident type) is missing. 

In [52]:
# Find protocols with NaturaIncidente blank (including empty strings and NaN)
natura_incidente_blank_mask = df["NaturaIncidente"].isna() | (
    df["NaturaIncidente"] == "")
natura_incidente_blank_protocols = df[natura_incidente_blank_mask]["Protocollo"].unique(
)

print(f"Found {len(natura_incidente_blank_protocols)} protocols with blank NaturaIncidente")

Found 0 protocols with blank NaturaIncidente


The accidents with 'NaturaIncidente' missing have already been deleted in steps 1 to 4

DATA CLEANING - DELETE AS MANY ACCIDENTS AS POSSIBLE - FIND 'TIPOPERSONA' INCONSISTENCIES

6) Identify accidents with inconsistent 'TipoPersona' numbers (blank or missing 'TipoPersona' values will make it difficult to understand the dynamics of what happened) 

In [53]:
analyse_column('TipoPersona')

,Count,% per category,Unique Accidents
TipoPersona,,,
Pedone,19730,47.77,18248
Conducente,17297,41.88,17297
Passeggero,3669,8.88,2932
<NA>,598,1.45,507
Pedone sconosciuto,4,0.01,4
Passeggero Istruttore,2,0.00,2


598 rows (507 accidents) have unidentified parties involved.

In [54]:
# Protocols with blank / null TipoPersona
mask = df["TipoPersona"].fillna("") == ""
blank_tipopersona_protocols = df.loc[mask, "Protocollo"].unique()

print(f"Protocols with blank TipoPersona: {len(blank_tipopersona_protocols)}")
print(f"Rows with blank TipoPersona: {mask.sum()}")

Protocols with blank TipoPersona: 507
Rows with blank TipoPersona: 598


Now we will view some of these accidents (blanks in 'TipoPersona') to try to understand why the 'TipoPersona' is missing. 

We need to look at all rows involved in the accident, so we select the protocols identified earlier.

In [55]:
df.loc[df["Protocollo"].isin(
    blank_tipopersona_protocols), df.columns[:6]].head(15)

,Protocollo,Progressivo,TipoPersona,TipoVeicolo,StatoVeicolo,NaturaIncidente
25889,4127572,<NA>,Pedone,,,Investimento di pedone
25890,4127572,1,Conducente,Motociclo a solo,In marcia / fermata / arresto,Investimento di pedone
25891,4127572,2,,Autovettura privata,Sosta,Investimento di pedone
26027,4136684,<NA>,Pedone,,,Investimento di pedone
26028,4136684,1,,Autovettura privata,In marcia / fermata / arresto,Investimento di pedone
26045,4138867,<NA>,Pedone,,,Investimento di pedone
26046,4138867,<NA>,Pedone,,,Investimento di pedone
26047,4138867,1,,Motociclo a solo,In marcia / fermata / arresto,Investimento di pedone
26081,4141181,<NA>,Pedone,,,Investimento di pedone
26082,4141181,1,Conducente,Motociclo con passeggero,In marcia / fermata / arresto,Investimento di pedone


Accidents where the 'TipoPersona' is missing seem to be of these types:

TYPE i)
A vehicle with a missing driver who is:
- StatoVeicolo: Allontanatosi
- StatoVeicolo: In marcia / fermata / arresto

TYPE ii)
A vehicle with a driver who is 'Conducente', plus:
- StatoVeicolo: Sosta and Progressivo > 1

TYPE i)
We need to look for the accidents where there is only one 'TipoPersona' blank. This fills in the blank 'TipoPersona' with 'Conducente'. 

In [56]:
print(
    f"Protocols with blank TipoPersona before: {len(blank_tipopersona_protocols)}")
print(f"Rows with blank TipoPersona before: {mask.sum()}")

# 1) Count blanks per protocol
blank = df["TipoPersona"].fillna("") == ""
blank_counts = df.groupby("Protocollo")["TipoPersona"].apply(
    lambda s: (s.fillna("") == "").sum())

# 2) Protocols with exactly one blank
one_blank_protocols = blank_counts[blank_counts == 1].index

# 3) Rows that are that single blank AND have allowed StatoVeicolo
allowed = {'Allontanatosi', 'In marcia / fermata / arresto'}
fix_rows = (
    blank
    & df["Protocollo"].isin(one_blank_protocols)
    & df["StatoVeicolo"].isin(allowed)
)

# 4) Fix them
df.loc[fix_rows, "TipoPersona"] = "Conducente"

# 5) Recompute summary
blank_after = df["TipoPersona"].fillna("") == ""

print(
    f"Protocols with blank TipoPersona after: {df.loc[blank_after, 'Protocollo'].nunique()}")
print(f"Rows with blank TipoPersona after: {blank_after.sum()}")

Protocols with blank TipoPersona before: 507
Rows with blank TipoPersona before: 598


Protocols with blank TipoPersona after: 163
Rows with blank TipoPersona after: 254


We have almost halved the number of rows to examine. 

TYPE ii)
We need to look for rows where there is one or more vehicles which have 'StatoVeicolo' is 'Sosta'.
Where there is a stopped vehicle involved, we will put a 1 in a new 'parked_vehicle_involved' column.

Let's first update the blank 'TipoPersona' protocols:

In [57]:
# Introduce parked_vehicle_involved column
if 'parked_vehicle_involved' not in df.columns:
    df['parked_vehicle_involved'] = 0

# What counts as "blank" TipoPersona?


def is_blank_tipopersona(s):
    return s.isna() | s.astype(str).str.strip().eq('')


# Initial count before cleaning
blank_mask_before = is_blank_tipopersona(df['TipoPersona'])
protocols_before = df.loc[blank_mask_before, 'Protocollo'].nunique()
rows_before = int(blank_mask_before.sum())

print(f"Protocols with blank TipoPersona BEFORE: {protocols_before}")
print(f"Rows with blank TipoPersona BEFORE: {rows_before}")

# Identify "blank TipoPersona" + "Sosta" rows to delete
sosta_blank_mask = is_blank_tipopersona(
    df['TipoPersona']) & (df['StatoVeicolo'] == 'Sosta')

# Protocols that have rows where the type of person is blank and the state of vehicle is parked (sosta)
protocols_to_mark = df.loc[sosta_blank_mask, 'Protocollo'].unique()
rows_deleted = int(sosta_blank_mask.sum())  # Count the rows to be deleted

# For those protocols, let 'parked_vehicle_involved' = 1
if len(protocols_to_mark):
    df.loc[df['Protocollo'].isin(protocols_to_mark),
           'parked_vehicle_involved'] = 1

# Delete the Sosta+blank rows and reset index
df = df.loc[~sosta_blank_mask].reset_index(drop=True)

print(
    f"\nProtocols where blank 'Sosta' rows were removed: {len(protocols_to_mark)}")
print(f"Rows deleted: {rows_deleted}")

# Count of protocols and rows after cleaning
blank_mask_after = is_blank_tipopersona(df['TipoPersona'])
protocols_after = df.loc[blank_mask_after, 'Protocollo'].nunique()
rows_after = int(blank_mask_after.sum())

print(f"\nProtocols with blank TipoPersona AFTER: {protocols_after}")
print(f"Rows with blank TipoPersona AFTER: {rows_after}")
print(f"DataFrame shape AFTER processing: {df.shape}")

Protocols with blank TipoPersona BEFORE: 163
Rows with blank TipoPersona BEFORE: 254

Protocols where blank 'Sosta' rows were removed: 162
Rows deleted: 239

Protocols with blank TipoPersona AFTER: 14
Rows with blank TipoPersona AFTER: 15
DataFrame shape AFTER processing: (41061, 37)


Now we are down to 15 rows of blank 'TipoPersona' data. Let us examine the protocols including those rows:

In [58]:
# Find protocols with at least one blank TipoPersona
protocols_with_blank = df.loc[df['TipoPersona'] == "", 'Protocollo'].unique()

# Show ALL rows of those protocols, limited to the first 6 columns
df.loc[df['Protocollo'].isin(protocols_with_blank), df.columns[:6]]

,Protocollo,Progressivo,TipoPersona,TipoVeicolo,StatoVeicolo,NaturaIncidente
27438,4290029,<NA>,Pedone,,,Investimento di pedone
27439,4290029,1,,Autovettura privata,In marcia / fermata / arresto,Investimento di pedone
27649,4315836,<NA>,Pedone,,,Veicolo in marcia contro veicolo in sosta
27650,4315836,3,,Autovettura privata,In marcia / fermata / arresto,Veicolo in marcia contro veicolo in sosta
27800,4332151,<NA>,Pedone,,,Investimento di pedone
27801,4332151,1,,Autovettura privata,Allontanatosi,Investimento di pedone
28307,4412048,<NA>,Pedone,,,Investimento di pedone
28308,4412048,2,,Autovettura privata,Allontanatosi,Investimento di pedone
29003,4499372,<NA>,Pedone,,,Investimento di pedone
29004,4499372,1,,Autovettura privata,In marcia / fermata / arresto,Investimento di pedone


These are accidents which previously had more than one 'TipoPersona' missing. Most of these involved parked vehicles, which have been removed, leaving the pedestrian and a driver (blank). 

Where the accident involves more than one different vehicle (not parked vehicles), we need to eliminate the accident from our list. 

Where the accident involves a pedestrian and one vehicle, we fill in 'Conducente' for the vehicle. 

In [59]:
# --- Fresh counts BEFORE ---
blank_mask_before = is_blank_tipopersona(df["TipoPersona"])
protocols_before = df.loc[blank_mask_before, "Protocollo"].nunique()
rows_before = int(blank_mask_before.sum())

print(f"Protocols with blank TipoPersona BEFORE: {protocols_before}")
print(f"Rows with blank TipoPersona BEFORE: {rows_before}")

# --- Work only on protocols with blanks ---
subset = df[blank_mask_before]

# Count how many distinct Progressivo per protocol (ignoring NaN)
progressivo_counts = (
    subset[subset["Progressivo"].notna()]
    .groupby("Protocollo")["Progressivo"]
    .nunique()
)

# Classify protocols
protocols_to_fix = progressivo_counts[progressivo_counts == 1].index
protocols_to_delete = progressivo_counts[progressivo_counts >= 2].index

# --- Apply fixes: set blank TipoPersona = 'Conducente' ---
fix_mask = (
    df["Protocollo"].isin(protocols_to_fix) &
    is_blank_tipopersona(df["TipoPersona"]) &
    df["Progressivo"].notna()
)
df.loc[fix_mask, "TipoPersona"] = "Conducente"

# --- Delete ambiguous protocols ---
df = df.loc[~df["Protocollo"].isin(protocols_to_delete)].reset_index(drop=True)

print(f"\nDeleted {len(protocols_to_delete)} protocols")
print(f"Fixed {len(protocols_to_fix)} protocols")
print(f"\nDataFrame shape AFTER cleaning: {df.shape}")

# --- Fresh counts AFTER ---
blank_mask_after = is_blank_tipopersona(df["TipoPersona"])
protocols_after = df.loc[blank_mask_after, "Protocollo"].nunique()
rows_after = int(blank_mask_after.sum())

print(f"\nProtocols with blank TipoPersona AFTER: {protocols_after}")
print(f"Rows with blank TipoPersona AFTER: {rows_after}")

Protocols with blank TipoPersona BEFORE: 14
Rows with blank TipoPersona BEFORE: 15

Deleted 1 protocols
Fixed 13 protocols

DataFrame shape AFTER cleaning: (41058, 37)

Protocols with blank TipoPersona AFTER: 0
Rows with blank TipoPersona AFTER: 0


Now every 'TipoPersona' has a value.

In [60]:
analyse_column('TipoPersona')

,Count,% per category,Unique Accidents
TipoPersona,,,
Pedone,19729,48.05,18247
Conducente,17654,43.00,17644
Passeggero,3669,8.94,2932
Pedone sconosciuto,4,0.01,4
Passeggero Istruttore,2,0.00,2


However, there are still accidents where more than one vehicle is involved (detectable by two different 'Progressivo' numbers).

In [61]:
# Count unique Progressivo values per protocol
progressivo_counts = df.groupby("Protocollo")["Progressivo"].nunique()

# Protocols with 2 or more distinct Progressivo values
multiple_progressivo_protocols = progressivo_counts[progressivo_counts >= 2].index.tolist(
)

print(f"Found {len(multiple_progressivo_protocols)} protocols with ≥ 2 unique Progressivo values")
if multiple_progressivo_protocols:
    print("Examples:", multiple_progressivo_protocols[:5])
else:
    print("No examples to show")

Found 32 protocols with ≥ 2 unique Progressivo values
Examples: [1934916, 1940340, 2232833, 2297125, 2308448]


In [62]:
df.loc[df['Protocollo'].isin(multiple_progressivo_protocols), df.columns[:6]]

,Protocollo,Progressivo,TipoPersona,TipoVeicolo,StatoVeicolo,NaturaIncidente
487,1934916,<NA>,Pedone,,,Fuoriuscita dalla sede stradale
488,1934916,1,Conducente,Autovettura privata,In marcia / fermata / arresto,Fuoriuscita dalla sede stradale
489,1934916,1,Passeggero,Autovettura privata,In marcia / fermata / arresto,Fuoriuscita dalla sede stradale
490,1934916,2,Passeggero,Autovettura privata,Sosta,Fuoriuscita dalla sede stradale
1179,1940340,<NA>,Pedone,,,Veicolo in marcia contro veicolo fermo
...,...,...,...,...,...,...
38709,5814342,1,Conducente,Autovettura privata,In marcia / fermata / arresto,Tamponamento
38710,5814342,2,Conducente,Motociclo a solo,Allontanatosi,Tamponamento
39696,5922338,<NA>,Pedone,,,Investimento di pedone
39697,5922338,1,Conducente,Autovettura privata,In marcia / fermata / arresto,Investimento di pedone


DATA CLEANING: PARKED VEHICLES

This time, we have rows where a vehicle is in 'Sosta' (parked) but this time the 'TipoPersona' is not blank, but instead marked as 'Conducente' or 'Passeggero'.

In [63]:
# --- BEFORE: protocols with ≥2 unique Progressivo (i.e., multiple vehicles/participants) ---
progressivo_counts_before = df.groupby(
    "Protocollo")["Progressivo"].nunique(dropna=True)
multi_proto_before = progressivo_counts_before[progressivo_counts_before >= 2].index
print(f"Protocols with multiple vehicles BEFORE: {len(multi_proto_before)}")

# --- Identify 'Sosta' rows within those protocols ---
stato_is_sosta = df["StatoVeicolo"].fillna("").eq("Sosta")
sosta_mask = df["Protocollo"].isin(multi_proto_before) & stato_is_sosta

# Protocols that actually have 'Sosta' rows
protocols_with_sosta = df.loc[sosta_mask, "Protocollo"].unique()
print(f"Protocols with Sosta to process: {len(protocols_with_sosta)}")

# Mark parked_vehicle_involved = 1 for those protocols
if len(protocols_with_sosta):
    df.loc[df["Protocollo"].isin(
        protocols_with_sosta), "parked_vehicle_involved"] = 1

# Delete all Sosta rows in those protocols
rows_deleted = int(sosta_mask.sum())
df = df.loc[~sosta_mask].reset_index(drop=True)

print(
    f"\nProcessed {len(protocols_with_sosta)} protocols with Sosta deletions")
print(f"Deleted {rows_deleted} rows")

# --- AFTER: recompute multiple-Progressivo protocols ---
progressivo_counts_after = df.groupby(
    "Protocollo")["Progressivo"].nunique(dropna=True)
multi_proto_after = progressivo_counts_after[progressivo_counts_after >= 2].index

print(f"\nProtocols with multiple vehicles AFTER: {len(multi_proto_after)}")
print(f"DataFrame shape AFTER processing: {df.shape}")

Protocols with multiple vehicles BEFORE: 32
Protocols with Sosta to process: 21

Processed 21 protocols with Sosta deletions
Deleted 29 rows

Protocols with multiple vehicles AFTER: 11
DataFrame shape AFTER processing: (41029, 37)


In [66]:
df.loc[df['Protocollo'].isin(multi_proto_after), df.columns[:6]]

,Protocollo,Progressivo,TipoPersona,TipoVeicolo,StatoVeicolo,NaturaIncidente
26626,4196048,<NA>,Pedone,,,Ribaltamento senza urto contro ostacolo fisso
26627,4196048,1,Conducente,Autovettura privata,In marcia / fermata / arresto,Ribaltamento senza urto contro ostacolo fisso
26628,4196048,2,Conducente,Quadriciclo leggero,In marcia / fermata / arresto,Ribaltamento senza urto contro ostacolo fisso
29615,4600141,<NA>,Pedone,,,Scontro laterale fra veicoli in marcia
29616,4600141,1,Conducente,Autobus di linea,In marcia / fermata / arresto,Scontro laterale fra veicoli in marcia
29617,4600141,2,Conducente,Autovettura privata,In marcia / fermata / arresto,Scontro laterale fra veicoli in marcia
30786,4726875,<NA>,Pedone,,,Investimento di pedone
30787,4726875,1,Conducente,Autovettura privata,Allontanatosi,Investimento di pedone
30788,4726875,2,Conducente,Autovettura privata,In marcia / fermata / arresto,Investimento di pedone
30789,4726875,2,Passeggero,Autovettura privata,In marcia / fermata / arresto,Investimento di pedone


All of these accidents involve multiple vehicles, so need to be removed.

In [67]:
# Delete all rows where Protocollo is in multiple_vehicle_protocols
df = df[~df["Protocollo"].isin(
    multi_proto_after)].reset_index(drop=True)

print(f"Deleted protocols: {len(multi_proto_after)}")
print(f"New dataframe shape: {df.shape}")

Deleted protocols: 11
New dataframe shape: (40989, 37)


In [68]:
analyse_column('StatoVeicolo')

C:\Users\lucyq\AppData\Local\Temp\ipykernel_7692\2466847765.py:18: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  cleaned_column = df[column_name].replace('', pd.NA)


,Count,% per category,Unique Accidents
StatoVeicolo,,,
In marcia / fermata / arresto,20947,51.10,17324
NaN,19721,48.11,18236
Allontanatosi,318,0.78,313
Sosta,3,0.01,3


In [70]:
protocols_with_sosta = df[df['StatoVeicolo'] == 'Sosta']['Protocollo'].unique()
df.loc[df['Protocollo'].isin(protocols_with_sosta), df.columns[:6]]

,Protocollo,Progressivo,TipoPersona,TipoVeicolo,StatoVeicolo,NaturaIncidente
10099,2336569,<NA>,Pedone,,,Investimento di pedone
10100,2336569,<NA>,Pedone,,,Investimento di pedone
10101,2336569,2,Passeggero,Autovettura privata,Sosta,Investimento di pedone
10610,2373364,<NA>,Pedone,,,Investimento di pedone
10611,2373364,1,Passeggero,Autovettura privata,Sosta,Investimento di pedone
25296,4061250,<NA>,Pedone,,,Veicolo in marcia contro veicolo in sosta
25297,4061250,1,Passeggero,Autovettura privata,Sosta,Veicolo in marcia contro veicolo in sosta


In these accidents, we have no information about a driver. We will delete these protocols.

In [71]:
df = df[~df["Protocollo"].isin(
    protocols_with_sosta)].reset_index(drop=True)

print(f"Deleted protocols: {len(protocols_with_sosta)}")
print(f"New dataframe shape: {df.shape}")

Deleted protocols: 3
New dataframe shape: (40982, 37)


DATA CLEANING: PASSENGERS

In [72]:
# Count distinct Progressivo values per protocol (ignoring NaN)
progressivo_counts = df.groupby(
    "Protocollo")["Progressivo"].nunique(dropna=True)

# Keep only protocols with more than one distinct Progressivo
multiple_progressivo_summary = progressivo_counts[progressivo_counts > 1]

print(
    f"Accidents with multiple different Progressivo values: {len(multiple_progressivo_summary)}")

Accidents with multiple different Progressivo values: 0


DATA CLEANING: DELETING COLUMNS WITH SEVERELY LIMITED INFORMATION

In [73]:
analyse_column('CinturaCascoUtilizzato')

C:\Users\lucyq\AppData\Local\Temp\ipykernel_7692\2466847765.py:18: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  cleaned_column = df[column_name].replace('', pd.NA)


,Count,% per category,Unique Accidents
CinturaCascoUtilizzato,,,
NaN,24859,60.66,18233
Non accertato,14467,35.30,12081
Utilizzato,1366,3.33,1186
Esente,220,0.54,203
Non utilizzato,70,0.17,69


'CinturaCascoUtilizzato': 61% have missing values, although this would be normal if the row is for a pedestrian. 35% were not checked by the police; this seems to be a column which can be dropped as in the majority of cases, we can not use it to ascertain anything about the accident(s). 

In [74]:
analyse_column('Airbag')

C:\Users\lucyq\AppData\Local\Temp\ipykernel_7692\2466847765.py:18: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  cleaned_column = df[column_name].replace('', pd.NA)


,Count,% per category,Unique Accidents
Airbag,,,
NaN,29556,72.12,18233
Inesploso,9393,22.92,7773
Inesistente,1978,4.83,1657
Esploso,55,0.13,51


'Airbag': 72% have missing values, although this would be normal if the vehicle is two-wheeled or if the row is for a pedestrian. The values in this column might help us understand the violence of the impact (as far as the vehicle is concerned), but this can be ascertained from the injuries to the pedestrian. Therefore we can drop this column. 

In [75]:
df = df.drop(['Airbag', 'CinturaCascoUtilizzato'], axis=1)
df.shape

(40982, 35)

In [76]:
df.to_parquet('004_data_only_pedestrians.parquet', index=False)

In [77]:
df.to_csv('004_data_only_pedestrians.csv', index=False)

In [ ]:
metadata = {
    'notebook': '004 Data cleaning.ipynb',
    'step': 'data cleaning → keep pedestrian accidents + protocol hygiene',
    'initial_parquet_file': 'data_no_duplicates.parquet',
    'initial_rows': 685_877,
    'initial_columns': 37,

    # Explicit column edits
    'columns_added': ['parked_vehicle_involved'],
    'columns_removed': ['Airbag', 'CinturaCascoUtilizzato'],
    'final_columns': 35,

    # Outputs
    'final_rows': 40_982,
    'final_parquet_file': '004_data_only_pedestrians.parquet',
    'final_csv_file': '004_data_only_pedestrians.csv',

    # High-level decisions taken in this notebook
    'decisions_made': [
        "Curated to a focused column set (priority, time, position, road type, injury).",
        "Removed 'Infortunio per caduta del veicolo' (falling from vehicle) accidents.",
        "Removed protocols with multiple drivers (TipoPersona == 'Conducente' ≥ 2).",
        "Kept only accidents involving at least one pedestrian (≈ accident protocols kept: 19,730).",
        "Defined and cleaned blank TipoPersona cases; audited before/after.",
        "Removed protocols with multiple vehicles after cleaning.",
        "Dropped Airbag/CinturaCascoUtilizzato (kept injury counters NUM_FERITI/NUM_MORTI/NUM_ILLESI)."
    ],

    # Repro-style deltas at each major filter (rows, columns)
    'filters_applied': [
        {
            'name': "remove_falling_from_vehicle",
            'before_shape': (685_877, 36),
            'after_shape': (682_844, 36),
            'rows_removed': 3_033,
            'share_%': 0.44
        },
        {
            'name': "remove_multi_driver_protocols",
            'before_shape': (682_844, 36),
            'after_shape': (151_400, 36),
            'rows_removed': 531_444,
            'share_%': 77.83
        },
        {
            'name': "keep_pedestrian_accidents",
            'before_shape': (151_400, 36),
            'after_shape': (41_300, 36),
            'rows_removed': 110_100,
            'share_%': 72.72,
            'accident_protocols_kept': 19_730
        },
        {
            'name': "blank_TipoPersona_cleanup + vehicle multiplicity audit",
            'before_shape': (41_300, 36),
            'after_shape': (41_029, 37),
            'protocols_with_multiple_vehicles_after': 11
        },
        {
            'name': "delete_protocols_with_multiple_vehicles",
            'before_shape': (41_029, 37),
            'after_shape': (40_989, 37),
            'protocols_removed': 11
        },
        {
            'name': "delete_additional_protocols",
            'before_shape': (40_989, 37),
            'after_shape': (40_982, 37),
            'protocols_removed': 3
        },
        {
            'name': "drop_columns_airbag_cintura",
            'before_shape': (40_982, 37),
            'after_shape': (40_982, 35)
        }
    ],

    # Quick roll-ups
    'rows_removed_total': 685_877 - 40_982,     # 644,895
    'rows_removed_share_%': 94.02,

    # Useful context for audits
    'duplicate_detection_context': "Downstream of 003 duplicate cleanup; this notebook focuses on protocol-level hygiene and pedestrian-only filtering.",
    'injury_counters_kept': ['NUM_FERITI', 'NUM_MORTI', 'NUM_ILLESI'],
}